In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from scipy import stats
from numpy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

In [ ]:
trees = pd.read_csv("biomass_model_dataset.csv")
species = pd.read_csv("FIATreeSpeciesCode.csv")

In [ ]:
species_codes = species[["SPCD","COMMON_NAME"]]
combined_df = trees.merge(species_codes,on='SPCD', how='left')
combined_df = combined_df.dropna(subset = "COMMON_NAME")
counts = combined_df["COMMON_NAME"].value_counts()
frequent_values = counts[counts >= 100].index
df_filtered = combined_df[combined_df["COMMON_NAME"].isin(frequent_values)]

In [ ]:
train = []
test = []
selection = []
tree = []
for i in range(len(df_filtered)):
  #ntrain = df_filtered["COMMON_NAME"].value_counts()[0] * 0.80
  #nst = df_filtered["COMMON_NAME"].value_counts()[0] - ntrain
  #nselection = nst/2
  #ntest = nselection
  if (df_filtered["COMMON_NAME"].iloc[i] == "loblolly pine"):
    tree.append(df_filtered.iloc[i])

In [ ]:
tree = df_filtered.loc[df_filtered['COMMON_NAME'] == "loblolly pine"]

#find last index for tarining, test, and selection index
train_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.80)
test_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.90)
selection_index = int(tree["COMMON_NAME"].value_counts()[0])

#put in new dataframes
train = pd.DataFrame(tree.iloc[:train_index])
test = pd.DataFrame(tree.iloc[train_index:test_index])
selection = pd.DataFrame(tree.iloc[test_index:selection_index])


In [ ]:
train_list=[]
test_list=[]
selection_list=[]

for species_name in df_filtered['COMMON_NAME'].unique():
    tree = df_filtered.loc[df_filtered['COMMON_NAME'] == species_name]
    #find last index for tarining, test, and selection index
    train_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.80)
    test_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.90)
    selection_index = int(tree["COMMON_NAME"].value_counts()[0])

    #put in new dataframes
    train = pd.DataFrame(tree.iloc[:train_index])
    test = pd.DataFrame(tree.iloc[train_index:test_index])
    selection = pd.DataFrame(tree.iloc[test_index:selection_index])

    train_list.append(train)
    test_list.append(test)
    selection_list.append(selection)

# Combine all species
train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)
selection_df = pd.concat(selection_list, ignore_index=True)

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_squared_log_error


# Metrics storage
performance = []

# Loop through each species
for species in train_df["COMMON_NAME"].unique():

    # Filter datasets
    train_species = train_df[train_df["COMMON_NAME"] == species]
    test_species = test_df[test_df["COMMON_NAME"] == species]

    # Skip if too few samples
    if len(train_species) < 3 or len(test_species) < 1:
        print(f"Skipping {species} (not enough data)")
        continue

    # Features and target
    X_train = train_species[['SPCD', 'DO_BH', 'HT_TOT']]
    y_train = train_species['TT_DW_CRM']

    X_test = test_species[['SPCD', 'DO_BH', 'HT_TOT']]
    y_test = test_species['TT_DW_CRM']

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Try k = 2 through 10
    for k in range(2, 11):

        model = KNeighborsRegressor(n_neighbors=k)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)

        # Compute metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)

        # RMSLE — must ensure predictions > 0
        y_pred_adj = np.maximum(y_pred, 1e-5)
        y_test_adj = np.maximum(y_test, 1e-5)
        rmsle = np.sqrt(mean_squared_log_error(y_test_adj, y_pred_adj))

        # Save metrics
        performance.append({
            "Species": species,
            "k": k,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "RMSLE": rmsle
        })

# Convert to dataframe
performance_df = pd.DataFrame(performance)
performance_df


,Species,k,MSE,RMSE,MAE,RMSLE
0,balsam fir,2,15832.049590,125.825473,34.801913,0.119574
1,balsam fir,3,16809.715674,129.652288,36.650638,0.123746
2,balsam fir,4,18111.360940,134.578456,38.747261,0.133460
3,balsam fir,5,21026.044395,145.003601,41.209530,0.144063
4,balsam fir,6,21741.872898,147.451256,40.787623,0.142856
...,...,...,...,...,...,...
895,sugarberry,6,1051.773141,32.431052,20.969306,0.565727
896,sugarberry,7,1393.100286,37.324259,25.037024,0.637629
897,sugarberry,8,1455.022421,38.144756,26.680208,0.665159
898,sugarberry,9,1358.161657,36.853245,21.021667,0.685162
